<a href="https://colab.research.google.com/github/betulbilhan2/LLM-Augmentation-Fidelity/blob/main/NB02_Synthetic_Generation_BT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Synthetic Generation - Duplication Control & Back-Translation

**Goal:**
1. Generate the **Duplication Control (E1)** group by making a 1:1 exact copy of the original train splits.
2. Generate the **Back-Translation (E2)** group using MarianMT (`TR -> EN -> TR`) for a 1:1 synthetic data generation.
3. Apply this to all 10 seeds across both `low` and `normal` resource levels.


In [1]:
# Install necessary libraries
!pip install transformers sentencepiece accelerate pandas pyyaml

import os
import sys
import yaml
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from google.colab import drive
import logging

# Mount Drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/tr_augmentation_project'

# Setup logging
log_file_path = os.path.join(PROJECT_ROOT, "logs", "nb02_run.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file_path, encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ],
    force=True
)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Load Experiment Config
config_path = os.path.join(PROJECT_ROOT, "configs", "experiment_config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

logger.info("Configuration loaded.")


Mounted at /content/drive
2026-08-31 10:45:52,133 - INFO - Configuration loaded.


In [2]:
# Initialize Models for Back-Translation
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}. (If CPU, normal resource level might take a while. Enable GPU in Colab: Runtime -> Change runtime type)")

logger.info("Loading TR->EN model...")
model_tr_en_name = "Helsinki-NLP/opus-mt-tr-en"
tokenizer_tr_en = MarianTokenizer.from_pretrained(model_tr_en_name)
model_tr_en = MarianMTModel.from_pretrained(model_tr_en_name).to(device)

logger.info("Loading EN->TR model...")
model_en_tr_name = "Helsinki-NLP/opus-mt-tc-big-en-tr"
tokenizer_en_tr = MarianTokenizer.from_pretrained(model_en_tr_name)
model_en_tr = MarianMTModel.from_pretrained(model_en_tr_name).to(device)

def back_translate(texts, batch_size=16):
    """Translates a list of Turkish texts to English, and back to Turkish."""
    # TR -> EN
    en_texts = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer_tr_en(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            translated = model_tr_en.generate(**inputs)
        en_texts.extend([tokenizer_tr_en.decode(t, skip_special_tokens=True) for t in translated])

    # EN -> TR
    tr_texts = []
    for i in range(0, len(en_texts), batch_size):
        batch = en_texts[i:i+batch_size]
        inputs = tokenizer_en_tr(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            translated = model_en_tr.generate(**inputs)
        tr_texts.extend([tokenizer_en_tr.decode(t, skip_special_tokens=True) for t in translated])

    return tr_texts


2026-08-31 10:45:52,282 - INFO - Using device: cuda. (If CPU, normal resource level might take a while. Enable GPU in Colab: Runtime -> Change runtime type)
2026-08-31 10:45:52,289 - INFO - Loading TR->EN model...
2026-08-31 10:45:52,851 - INFO - HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-31 10:45:52,971 - INFO - HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-tr-en/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-31 10:45:52,972 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-31 10:45:52,987 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tr-en/19c65427cc2af5f191337d4899e0348c4af25902/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-31 10:45:53,001 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tr-en/19c65427cc2af5f191337d4899e0348c4af25902/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

2026-08-31 10:45:53,147 - INFO - HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-tr-en/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-31 10:45:53,265 - INFO - HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-tr-en/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-31 10:45:53,385 - INFO - HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-tr-en/resolve/main/source.spm "HTTP/1.1 307 Temporary Redirect"
2026-08-31 10:45:53,397 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tr-en/19c65427cc2af5f191337d4899e0348c4af25902/source.spm "HTTP/1.1 200 OK"
2026-08-31 10:45:53,410 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tr-en/19c65427cc2af5f191337d4899e0348c4af25902/source.spm "HTTP/1.1 200 OK"


source.spm:   0%|          | 0.00/840k [00:00<?, ?B/s]

...

2026-08-31 10:46:14,513 - INFO - HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-tc-big-en-tr/commits/main "HTTP/1.1 200 OK"
2026-08-31 10:46:14,649 - INFO - HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-tc-big-en-tr/discussions?p=0 "HTTP/1.1 200 OK"
2026-08-31 10:46:14,846 - INFO - HTTP Request: GET https://huggingface.co/api/models/Helsinki-NLP/opus-mt-tc-big-en-tr/commits/refs%2Fpr%2F10 "HTTP/1.1 200 OK"
2026-08-31 10:46:14,974 - INFO - HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr/resolve/refs%2Fpr%2F10/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-08-31 10:46:15,108 - INFO - HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr/resolve/refs%2Fpr%2F10/model.safetensors "HTTP/1.1 302 Found"


model.safetensors: reconstructing file:   0%|          |  0.00B /  470MB            

model.safetensors: downloading bytes:           |  0.00B            

2026-08-31 10:46:31,142 - INFO - HTTP Request: HEAD https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-tr/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 10:46:31,157 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tc-big-en-tr/e539fc16a8a1a0ea5950eb339b595bfcce990e90/generation_config.json "HTTP/1.1 200 OK"
2026-08-31 10:46:31,172 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Helsinki-NLP/opus-mt-tc-big-en-tr/e539fc16a8a1a0ea5950eb339b595bfcce990e90/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [3]:
# Execute Generation Pipeline

DATA_SEEDS = config["seeds"]["data_seeds"]
LEVELS = ["low", "normal"]

for level in LEVELS:
    logger.info(f"--- Processing Resource Level: {level.upper()} ---")

    for seed in DATA_SEEDS:
        logger.info(f"  Seed {seed}/9 processing...")

        # Load Original Seed Data
        train_path = os.path.join(PROJECT_ROOT, f"01_splits/{level}/seed_{seed}/train_seed.csv")
        df_train = pd.read_csv(train_path)

        # 1. Duplication Control (E1)
        # Simply copying the original dataset to act as E1 (Quantity vs Quality Control)
        dup_dir = os.path.join(PROJECT_ROOT, f"02_augmented/{level}/duplication_control/seed_{seed}")
        os.makedirs(dup_dir, exist_ok=True)
        df_train.to_csv(os.path.join(dup_dir, "pool.csv"), index=False)

        # 2. Back-Translation (E2)
        bt_texts = back_translate(df_train['text'].tolist(), batch_size=32)
        df_bt = df_train.copy()
        df_bt['text'] = bt_texts

        bt_dir = os.path.join(PROJECT_ROOT, f"02_augmented/{level}/backtranslation/seed_{seed}")
        os.makedirs(bt_dir, exist_ok=True)
        df_bt.to_csv(os.path.join(bt_dir, "pool.csv"), index=False)

logger.info("✅ Synthetic Generation (Duplication Control & Back-Translation) completed successfully for all seeds and levels.")


2026-08-31 10:46:31,463 - INFO - --- Processing Resource Level: LOW ---
2026-08-31 10:46:31,465 - INFO -   Seed 0/9 processing...
2026-08-31 10:46:48,537 - INFO -   Seed 1/9 processing...
2026-08-31 10:47:29,075 - INFO -   Seed 2/9 processing...
2026-08-31 10:47:37,748 - INFO -   Seed 3/9 processing...
2026-08-31 10:47:48,025 - INFO -   Seed 4/9 processing...
2026-08-31 10:47:54,145 - INFO -   Seed 5/9 processing...
2026-08-31 10:48:05,219 - INFO -   Seed 6/9 processing...
2026-08-31 10:48:22,331 - INFO -   Seed 7/9 processing...
2026-08-31 10:48:31,300 - INFO -   Seed 8/9 processing...
2026-08-31 10:48:44,066 - INFO -   Seed 9/9 processing...
2026-08-31 10:48:52,304 - INFO - --- Processing Resource Level: NORMAL ---
2026-08-31 10:48:52,305 - INFO -   Seed 0/9 processing...
2026-08-31 10:52:44,106 - INFO -   Seed 1/9 processing...
2026-08-31 10:55:43,969 - INFO -   Seed 2/9 processing...
2026-08-31 10:57:15,339 - INFO -   Seed 3/9 processing...
2026-08-31 11:00:51,258 - INFO -   Seed 4

In [4]:
import logging
from google.colab import drive

# 1. Python'un bellekte tuttuğu logları zorla diske yazdırıyoruz
for handler in logging.root.handlers:
    handler.flush()

if 'logger' in globals():
    for handler in logger.handlers:
        handler.flush()

# 2. Colab üzerindeki önbelleği (cache) zorla Google Drive'a gönderip senkronize ediyoruz
drive.flush_and_unmount()
print("The writing process to Drive was successfully completed. You can check the Drive.")

The writing process to Drive was successfully completed. You can check the Drive.
